# Faruq-v3 - AF2_ORIENT paired confirmation

Ubah `SEED` menjadi 123 atau 2026. Jalankan satu seed per akun. Output tersimpan langsung ke folder proyek Drive, dapat resume, dan test tidak pernah dipulihkan.

In [ ]:
SEED = 123  # akun kedua: 2026
assert SEED in (123, 2026)
BRANCH = 'agent/af2-continuation-confirmation'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
import torch
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection')
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('GPU:',torch.cuda.get_device_name(0),'| SEED:',SEED,'| BRANCH:',BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
D0_REL=f'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed{SEED}/weights/best.pt'
required=(
 'bundles/faruq-development-v3-grouped.tar',
 D0_REL,
 'experiments/faruq-v3-af2-igem-paired-confirmation-v1/val_reports/af2_igem_paired_confirmation.json',
 'experiments/faruq-v3-af2-isolated-seed42-v1/val_reports/AF2_ORIENT_seed42_result.json',
)
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=required)
ARCHIVE=require_project_artifact(PROJECT_ROOT,required[0])
D0=require_project_artifact(PROJECT_ROOT,D0_REL)
AF2_CONFIRM=require_project_artifact(PROJECT_ROOT,required[2])
SEED42_RESULT=require_project_artifact(PROJECT_ROOT,required[3])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'faruq_grouped_summary.json').is_file():
    if DATA.exists(): shutil.rmtree(DATA)
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file() and not (DATA/'test').exists()
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-af2-orient-paired-confirmation-v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('D0:',D0); print('OUTPUT:',OUTPUT)

In [ ]:
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_iso_arm',
 '--arm','AF2_ORIENT','--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),
 '--d0-checkpoint',str(D0),'--output-root',str(OUTPUT),'--seed',str(SEED),
 '--device','0','--latency-iterations','50','--authorize-training']
LOG=OUTPUT/f'AF2_ORIENT_seed{SEED}_run.log'
print('START/RESUME AF2_ORIENT seed',SEED,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
csv_path=OUTPUT/f'AF2_ORIENT/AF2_ORIENT_seed{SEED}/results.csv'
seen=None
while process.poll() is None:
    epochs=max(0,len(csv_path.read_text(errors='replace').splitlines())-1) if csv_path.is_file() else 0
    if epochs!=seen:
        print(f'AF2_ORIENT seed {SEED}: {epochs}/50 epoch tercatat',flush=True); seen=epochs
    time.sleep(30)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
    raise RuntimeError(f'AF2_ORIENT seed {SEED} gagal: {process.returncode}')
RESULT=OUTPUT/f'val_reports/AF2_ORIENT_seed{SEED}_result.json'
assert RESULT.is_file(),RESULT
payload=json.loads(RESULT.read_text())
print(json.dumps({k:payload['metrics'][k] for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')},indent=2))

In [ ]:
r123=OUTPUT/'val_reports/AF2_ORIENT_seed123_result.json'
r2026=OUTPUT/'val_reports/AF2_ORIENT_seed2026_result.json'
if r123.is_file() and r2026.is_file():
    decision=OUTPUT/'val_reports/af2_orient_paired_confirmation.json'
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_orient_confirmation_decision',
      '--af2-confirmation',str(AF2_CONFIRM),'--orient-seed42-result',str(SEED42_RESULT),
      '--orient-seed123-result',str(r123),'--orient-seed2026-result',str(r2026),'--output',str(decision)]
    subprocess.run(cmd,cwd=REPO,check=True)
    print('DECISION:',decision)
else:
    print('Menunggu arm pasangan:',{'123':r123.is_file(),'2026':r2026.is_file()})